In [93]:
#run this everytime an edit is made in utils.py so that we can use the helper functions in this Python Notebook
import importlib
import utils1
importlib.reload(utils1)

<module 'utils1' from '/Users/ivychan/finm-367/finm-367-groupa/ivy-work/HW 1/utils1.py'>

# Harvard's Endowment

## HBS Case
### *The Harvard Management Company and Inflation-Indexed Bonds*

$$
\newcommand{\mutarg}{\tilde{\boldsymbol{\mu}}^{\text{port}}}
\newcommand{\mux}{\tilde{\mu}}
\newcommand{\wEW}{\boldsymbol{\text{w}}^{\text{EW}}}
\newcommand{\wREG}{\boldsymbol{\text{w}}^{\text{REG}}}
\newcommand{\wRP}{\boldsymbol{\text{w}}^{\text{RP}}}
\newcommand{\wtan}{\boldsymbol{\text{w}}^{\text{tan}}}
$$

# 1. READING: HMC's Approach

### 1. 
**There are thousands of individual risky assets in which HMC can invest.  Explain why MV optimization across 1,000 securities is infeasible.**


MV optimization across 1,000 individual securities is infeasible becuase it requires estimating 1,000 expected returns and a 1,000 x 1,000 covariance matrix, totaling to 501,500 unique risk parameters. These estimates are difficult to obtain accurately and MV optimization is highly sensitive to estimation error. Therefore, HMC instead performs its optimization at the asset-class level which dramatically reduced the number of inputs and produced more stable portfolio allocations.

### 2.
**Rather than optimize across all securities directly, HMC runs a two-stage optimization.**
1. They build asset class portfolios with each one optimized over the securities of the specific asset class.  
2. HMC combines the asset-class portfolios into one total optimized portfolio.

**In order for the two-stage optimization to be a good approximation of the full MV-optimization on all assets, what must be true of the partition of securities into asset classes?**


For the two-stage optimization to approximate a full MV optimization, securities must be grouped into asset classes that are internally similar, particularly in how they covary with securities outside the class. Otherwise, optimizing within each class first could discard diversification opportunities that would have been selected by a full security-level optimization.


### 3.
**Should TIPS form a new asset class or be grouped into one of the other 11 classes?**

TIPS have a reasonable case for being treated as a separate asset class rather than simply grouped with nominal domestic bonds. Their inflation protection gives them a different source of risk and return, and therefore potentially different correlations with the rest of the portfolio. HMC's optimization assigned substantial weight to inflation-indexed bonds, and its proposed Policy Portfolio ultimately included a separate 7% TIPS allocation. However, our empirical analysis also shows that the strength of the case depends heavily on the assumed expected return for TIPS.

### 4. 
**Why does HMC focus on real returns when analyzing its portfolio allocation? Is this just a matter of scaling, or does using real returns versus nominal returns potentially change the MV solution?**

HMC focuses on real returns because Harvard ultimately cares about preserving and growing the endowment's purchasing power. Converting from nominal to real returns is not merely a scaling exercise: inflation affects the expected returns, variances, and covariances of different asset classes differently. Because the MV solution depends on both expected returns and the covariance matrix, using real rather than nominal returns can change the optimal portfolio.

### 5.
**The case discusses the fact that Harvard places bounds on the portfolio allocation rather than implementing whatever numbers come out of the MV optimization problem.**

**How might we adjust the stated optimization problem in the lecture notes to reflect the extra constraints Harvard is using in their bounded solutions given in Exhibits 5 and 6?**

Harvard's portfolio restrictions can be incorporated into the standard MV optimization by adding lower- and upper-bound constraints $L_i \leq w_i \leq U_i$ for each asset class. These constraints prevent the optimizer from taking extreme positions that may be mathematically optimal but institutionally impractical.


### 6. 
**Exhibits 5 shows zero allocation to domestic equities and domestic bonds across the entire computed range of targeted returns, (5.75% to 7.25%). Conceptually, why is the constraint binding in all these cases? What would the unconstrained portfolio want to do with those allocations and why?**

The 0% lower bounds on domestic equity and domestic bonds are binding because the unconstrained optimizer would prefer allocations below zero—that is, short positions. Given HMC's capital-market assumptions, these assets do not provide sufficiently attractive expected returns and diversification benefits relative to the alternatives. Because Exhibit 5 prohibits shorting these asset classes, the optimizer pushes them as far down as permitted, to exactly 0%.

### 7.
**Exhibit 6 changes the constraints, (tightening them in most cases.) How much deterioration do we see in the mean-variance tradeoff that Harvard achieved?**

Tightening the portfolio constraints causes a noticeable but relatively modest deterioration in the mean-variance tradeoff. The Sharpe ratio falls from approximately 0.38 in Exhibit 5 to about 0.35–0.36 in Exhibit 6. For example, at a 6.50% expected real return, annualized volatility increases from 7.83% to 8.42%, while at a 7.25% target it increases from 9.88% to 10.61%. Thus, the tighter institutional constraints prevent HMC from reaching the unconstrained efficient frontier, but the resulting loss in risk-adjusted performance is not enormous.

***

# 2. Mean-Variance Optimization

### Data
You will need the file in the github repo, `data/multi_asset_etf_data.xlsx`.
- The time-series data gives monthly returns for the 11 asset classes and a short-term Treasury-bill fund return, (`SHV`.)
- The case does not give time-series data, so this data has been compiled outside of the case, and it intends to represent the main asset classes under consideration via various ETFs. For details on the specific securities/indexes, check the “Info” tab of the data.

### Excess Returns
- We consider `SHV` as the risk-free asset.
- We are going to analyze the problem in terms of **excess** returns, where `SHV` has been subtracted from the other columns.
- The risk-free rate changes over time, but the assumption is that investors know it’s value one-period ahead of time. Thus, at any given point in time, it is a risk-free rate for the next period. (This is often discussed as the "bank account" or "money market account" in other settings.)

### Adjustment
For ease of analysis, drop `QAI` from the dataset. Analyze the remaining 10 assets.

### Not Considered
- These are nominal returns-they are not adjusted for inflation, and in our calculations we are not making any adjustment for inflation.
- The exhibit data that comes via Harvard with the case is unnecessary for our analysis.

### Format
In the questions below, **annualize the statistics** you report.
- Annualize the mean of monthly returns with a scaling of 12.
- Annualize the volatility of monthly returns with a scaling of $\sqrt{12}$
- Note that we are not scaling the raw timeseries data, just the statistics computed from it (mean, vol, Sharpe).

In [94]:
import numpy as np
import pandas as pd

pd.options.display.float_format = "{:.4f}".format
path = "multi_asset_etf_data.xlsx"

excess_returns = pd.read_excel(path, sheet_name = "excess returns", index_col = 0)
excess_returns = excess_returns.drop(columns = "QAI")
display(excess_returns)

,BWX,DBC,EEM,EFA,HYG,IEF,IYR,PSP,SPY,TIP
Date,,,,,,,,,,
2011-02-28,0.0065,0.0416,-0.0005,0.0354,0.0143,-0.0022,0.0451,0.0401,0.0346,0.0071
2011-03-31,0.0083,0.0264,0.0627,-0.0241,0.0003,-0.0017,-0.0111,0.0165,-0.0001,0.0117
2011-04-30,0.0483,0.0450,0.0268,0.0557,0.0154,0.0179,0.0461,0.0581,0.0284,0.0232
2011-05-31,-0.0134,-0.0516,-0.0293,-0.0220,0.0014,0.0252,0.0102,-0.0415,-0.0111,0.0028
2011-06-30,-0.0003,-0.0428,-0.0095,-0.0123,-0.0059,-0.0052,-0.0312,-0.0428,-0.0170,0.0074
...,...,...,...,...,...,...,...,...,...,...
2026-03-31,-0.0480,0.1506,-0.0953,-0.0810,-0.0123,-0.0260,-0.0665,-0.0641,-0.0521,-0.0163
2026-04-30,0.0137,0.0713,0.1238,0.0505,0.0123,-0.0044,0.0824,0.0749,0.1021,0.0079
2026-05-31,-0.0054,-0.0550,0.0692,0.0213,0.0015,-0.0031,-0.0146,-0.0019,0.0497,-0.0010


### 1. Summary Statistics
* Calculate and display the mean and volatility of each asset’s excess return. (Recall we use volatility to refer to standard deviation.)
* Which assets have the best and worst Sharpe ratios? 

Recall that the Sharpe Ratio is simply the ratio of the mean-to-volatility of excess returns:

$$\text{sharpe ratio of investment }i = \frac{\mux_i}{\sigma_i}$$

Be sure to annualize all three statss (mean, vol, Sharpe).

In [103]:
ann_mean, ann_vol = utils1.annualize_stats(excess_returns.mean(), excess_returns.std())

summary_stats = pd.DataFrame({"Mean": ann_mean,
                              "Volatility": ann_vol,
                              "Sharpe": ann_mean / ann_vol})
display(summary_stats)

ranked_sharpes = (summary_stats["Sharpe"].sort_values(ascending = False).to_frame(name = "Ranked Sharpes"))
display(ranked_sharpes)

,Mean,Volatility,Sharpe
BWX,-0.0172,0.0819,-0.2096
DBC,0.0130,0.1723,0.0755
EEM,0.0444,0.1791,0.2481
EFA,0.0638,0.1486,0.4291
HYG,0.0345,0.0733,0.4705
IEF,0.0086,0.0621,0.1384
IYR,0.0701,0.1655,0.4235
PSP,0.0764,0.2113,0.3617
SPY,0.1270,0.1414,0.8979
TIP,0.0128,0.0497,0.2568


,Ranked Sharpes
SPY,0.8979
HYG,0.4705
EFA,0.4291
IYR,0.4235
PSP,0.3617
TIP,0.2568
EEM,0.2481
IEF,0.1384
DBC,0.0755
BWX,-0.2096


The asset with the best Sharpe is SPY with 0.8979, whereas the worst Sharpe is BWX with -0.2096

### 2. Descriptive Analysis
* Calculate the correlation matrix of the returns. Which pair has the highest correlation? And the lowest?
* How well have TIPS done in our sample? Have they outperformed domestic bonds? Foreign bonds?

In [96]:
correlation_matrix = excess_returns.corr()
display(correlation_matrix)

pairs = correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape, dtype = bool), k = 1)).stack() #unique pairs
ranked_correlation = pd.DataFrame({"Pair": [pairs.idxmax(), pairs.idxmin()],
                                   "Correlation": [pairs.max(), pairs.min()]}, index = ["Highest", "Lowest"])
display(ranked_correlation)

summary_stats.loc[["TIP", "IEF", "BWX"]]

,BWX,DBC,EEM,EFA,HYG,IEF,IYR,PSP,SPY,TIP
BWX,1.0000,0.1608,0.6117,0.6127,0.6004,0.5808,0.5550,0.5239,0.4435,0.6750
DBC,0.1608,1.0000,0.4388,0.4405,0.4188,-0.3088,0.2526,0.4202,0.3836,0.0843
EEM,0.6117,0.4388,1.0000,0.8120,0.6696,0.0515,0.5743,0.6974,0.6935,0.3839
EFA,0.6127,0.4405,0.8120,1.0000,0.7764,0.0629,0.7004,0.8616,0.8328,0.4025
HYG,0.6004,0.4188,0.6696,0.7764,1.0000,0.1927,0.7313,0.7983,0.7831,0.5404
IEF,0.5808,-0.3088,0.0515,0.0629,0.1927,1.0000,0.3218,0.0125,0.0113,0.7561
IYR,0.5550,0.2526,0.5743,0.7004,0.7313,0.3218,1.0000,0.7260,0.7435,0.5999
PSP,0.5239,0.4202,0.6974,0.8616,0.7983,0.0125,0.7260,1.0000,0.8722,0.3967
SPY,0.4435,0.3836,0.6935,0.8328,0.7831,0.0113,0.7435,0.8722,1.0000,0.3859
TIP,0.6750,0.0843,0.3839,0.4025,0.5404,0.7561,0.5999,0.3967,0.3859,1.0000


,Pair,Correlation
Highest,"(PSP, SPY)",0.8722
Lowest,"(DBC, IEF)",-0.3088


,Mean,Volatility,Sharpe
TIP,0.0128,0.0497,0.2568
IEF,0.0086,0.0621,0.1384
BWX,-0.0172,0.0819,-0.2096


TIPS performed well over the sample with an annualized excess return of 1.28% and a Sharpe ratio of 0.2568. It outperformed both domestic bonds (IEF) and foreign bonds (BWX) in terms of average excess return and Sharpe ratio while also exhibiting lower volatility.

### 3. The MV frontier.
* Compute and display the weights of the tangency portfolios: $\wtan$.
* Does the ranking of weights align with the ranking of Sharpe ratios?
* Compute the mean, volatility, and Sharpe ratio for the tangency portfolio corresponding to
$\wtan$.

In [97]:
tan_weights = utils1.tangency_weights(excess_returns)
display(tan_weights.to_frame("Tangency Weights"))

ranked_weights = tan_weights.sort_values(ascending = False)
weights_sharpes = pd.DataFrame({"Ranked Weights": ranked_weights.index,
                                "Tangency Weights": ranked_weights.values,
                                "Ranked Sharpe": ranked_sharpes.index,
                                "Sharpe": ranked_sharpes["Ranked Sharpes"].values})
display(weights_sharpes)

tan_mean, tan_vol, tan_sharpe = utils1.portfolio_stats(excess_returns, tan_weights)
tan_stats = pd.DataFrame({"Mean": [tan_mean],
                          "Volatility": [tan_vol],
                          "Sharpe": [tan_sharpe]}, index = ["Tangency Portfolio"])
display(tan_stats)

,Tangency Weights
BWX,-1.1797
DBC,0.0363
EEM,-0.0010
EFA,0.1848
HYG,0.2038
IEF,1.3188
IYR,-0.2237
PSP,-0.4106
SPY,1.2632
TIP,-0.1920


,Ranked Weights,Tangency Weights,Ranked Sharpe,Sharpe
0,IEF,1.3188,SPY,0.8979
1,SPY,1.2632,HYG,0.4705
2,HYG,0.2038,EFA,0.4291
3,EFA,0.1848,IYR,0.4235
4,DBC,0.0363,PSP,0.3617
5,EEM,-0.0010,TIP,0.2568
6,TIP,-0.1920,EEM,0.2481
7,IYR,-0.2237,IEF,0.1384
8,PSP,-0.4106,DBC,0.0755
9,BWX,-1.1797,BWX,-0.2096


,Mean,Volatility,Sharpe
Tangency Portfolio,0.1617,0.1096,1.4758


The ranking of tangency weights does not align with the ranking of individual Sharpe ratios. For example, SPY has the highest individual Sharpe ratio (0.8979) but IEF receives the largest portfolio weight despite having a relatively low individual Sharpe ratio (0.1384). This occurs because tangency portfolio weights depend not only on each asset’s individual risk and return but also on its covariance with the other assets and therefore its diversification benefits.

The tangency portfolio has an annualized mean excess return of 16.17%, annualized volatility of 10.96%, and an annualized Sharpe ratio of 1.4758. Its Sharpe ratio is substantially higher than that of any individual asset in the sample, illustrating the diversification benefit from combining assets based on their joint covariance structure.

### 4. TIPS
Assess how much the tangency portfolio (and performance) change if...
* TIPS are dropped completely from the investment set.
* The expected excess return to TIPS is adjusted to be 0.0012 higher than what the historic sample shows.

Based on the analysis, do TIPS seem to expand the investment opportunity set, implying that Harvard should consider them as a separate asset?

In [98]:
no_tips_excess_returns = excess_returns.drop(columns = "TIP")
no_tips_weights = utils1.tangency_weights(no_tips_excess_returns)
no_tips_mean, no_tips_vol, no_tips_sharpe = utils1.portfolio_stats(no_tips_excess_returns, no_tips_weights)

adjusted_mean = excess_returns.mean().copy()
adjusted_mean["TIP"] += 0.0012

adjusted_tips_weights = utils1.tangency_weights(excess_returns, adjusted_mean)

adjusted_tips_mean = adjusted_mean @ adjusted_tips_weights * 12
adjusted_tips_vol = (np.sqrt(adjusted_tips_weights.T @ excess_returns.cov() @ adjusted_tips_weights) * np.sqrt(12))
adjusted_tips_sharpe = adjusted_tips_mean / adjusted_tips_vol

tips_comparison = pd.DataFrame({"Mean": [tan_mean, no_tips_mean, adjusted_tips_mean],
                                "Volatility": [tan_vol, no_tips_vol, adjusted_tips_vol],
                                "Sharpe": [tan_sharpe, no_tips_sharpe, adjusted_tips_sharpe],
                                "TIP Weight": [tan_weights["TIP"], 0, adjusted_tips_weights["TIP"]]}, 
                                index = ["TIPS", "No TIPS", "TIPS mean + 0.0012"])
display(tips_comparison)

weights_comparison = pd.DataFrame({"TIPS": tan_weights,
                                   "No TIPS": no_tips_weights,
                                   "TIPS mean + 0.0012": adjusted_tips_weights}).fillna(0)
display(weights_comparison)

,Mean,Volatility,Sharpe,TIP Weight
TIPS,0.1617,0.1096,1.4758,-0.1920
No TIPS,0.1554,0.1054,1.4745,0.0000
TIPS mean + 0.0012,0.1327,0.0842,1.5749,1.2617


,TIPS,No TIPS,TIPS mean + 0.0012
BWX,-1.1797,-1.1375,-0.8602
DBC,0.0363,0.0258,-0.0433
EEM,-0.0010,-0.0062,-0.0402
EFA,0.1848,0.1881,0.2094
HYG,0.2038,0.1791,0.0172
IEF,1.3188,1.1616,0.1289
IYR,-0.2237,-0.2234,-0.2216
PSP,-0.4106,-0.4004,-0.3338
SPY,1.2632,1.2128,0.8819
TIP,-0.1920,0.0000,1.2617


Dropping TIPS from the investment set has very little effect on the tangency portfolio’s performance. 

The Sharpe ratio decreases only slightly from 1.4758 to 1.4745, and most of the portfolio weights remain similar. For example, SPY changes from 1.2632 to 1.2128 and BWX from -1.1797 to -1.1375. This suggests that, based on the historical sample, TIPS provide only a small incremental diversification benefit.

However, when the expected monthly excess return of TIPS is increased by 0.0012, the tangency portfolio changes substantially. The optimal TIP weight increases from -0.1920 to 1.2617, while several other positions change considerably—for example, IEF falls from 1.3188 to 0.1289 and SPY falls from 1.2632 to 0.8819. At the same time, the portfolio Sharpe ratio increases from 1.4758 to 1.5749, indicating a meaningful improvement in risk-adjusted performance.

Overall, the historical sample provides limited evidence that TIPS materially expand the investment opportunity set, since removing them barely changes the optimal portfolio or its Sharpe ratio. However, if Harvard expects TIPS to earn a higher excess return than their historical sample suggests, TIPS can materially alter and improve the investment opportunity set. Thus, the case for treating TIPS as a separate asset depends importantly on Harvard's forward-looking expected return for TIPS.

***

# 3. Allocations

* Continue with the same data file as the previous section.

* Suppose the investor has a targeted mean excess return (per month) of $\mutarg$ = 0.01.

Build the following portfolios:

#### Equally-weighted (EW)
Rescale the entire weighting vector to have target mean $\mutarg$. Thus, the $i$ element of the weight vector is,

$$\wEW_i = \frac{1}{n}$$

#### “Risk-parity” (RP)
Risk-parity is a term used in a variety of ways, but here we have in mind setting the weight of the portfolio to be proportional to the inverse of its full-sample variance estimate. Thus, the $i$ element of the weight vector is,

$$\wRP_i = \frac{1}{\sigma_i^2}$$

#### Mean-Variance (MV)
As described in `Section 2`.

In [106]:
TARGET_RETURN = 0.01

n = excess_returns.shape[1]
ew_weights = pd.Series(1 / n, index = excess_returns.columns)
ew_weights = utils1.scale_weights_to_target(ew_weights, excess_returns.mean(), TARGET_RETURN)

rp_weights = 1 / excess_returns.var()
rp_weights = utils1.scale_weights_to_target(rp_weights, excess_returns.mean(), TARGET_RETURN)

mv_weights = utils1.tangency_weights(excess_returns)
mv_weights = utils1.scale_weights_to_target(mv_weights, excess_returns.mean(),TARGET_RETURN)

allocation_weights = pd.DataFrame({"EW": ew_weights, 
                                   "RP": rp_weights, 
                                   "MV": mv_weights})
display(allocation_weights)


,EW,RP,MV
BWX,0.2769,0.6733,-0.8756
DBC,0.2769,0.1521,0.0269
EEM,0.2769,0.1408,-0.0007
EFA,0.2769,0.2045,0.1372
HYG,0.2769,0.8397,0.1513
IEF,0.2769,1.1700,0.9788
IYR,0.2769,0.1648,-0.1660
PSP,0.2769,0.1011,-0.3047
SPY,0.2769,0.2258,0.9375
TIP,0.2769,1.8244,-0.1425


### Comparing

In order to compare all these allocation methods, rescale each weight vector, such that it has targeted mean return of $\mutarg$.

* Calculate the performance of each of these portfolios over the sample.
* Report their mean, volatility, and Sharpe ratio. 
* How does performance compare across allocation methods?

In [107]:
ew_mean, ew_vol, ew_sharpe = utils1.portfolio_stats(excess_returns, ew_weights)
rp_mean, rp_vol, rp_sharpe = utils1.portfolio_stats(excess_returns, rp_weights)
mv_mean, mv_vol, mv_sharpe = utils1.portfolio_stats(excess_returns, mv_weights)

allocation_stats = pd.DataFrame({"Mean": [ew_mean, rp_mean, mv_mean],
                                 "Volatility": [ew_vol, rp_vol, mv_vol],
                                 "Sharpe": [ew_sharpe, rp_sharpe, mv_sharpe]}, index = ["EW", "RP", "MV"])
display(allocation_stats)

,Mean,Volatility,Sharpe
EW,0.1200,0.2753,0.4359
RP,0.1200,0.3261,0.3679
MV,0.1200,0.0813,1.4758


All three allocation methods are rescaled to achieve the same target monthly excess return of 1%, corresponding to an annualized mean excess return of 12%. Holding expected return constant makes the differences in volatility especially informative: the portfolio with the lowest volatility achieves the target return the most efficiently and consequently has the highest Sharpe ratio.

The Risk-Parity (RP) allocation is constructed using inverse variance so before rescaling it places relatively more weight on assets with lower individual volatility. This is intuitive from a risk-control perspective: rather than allocating equally across assets, RP reduces exposure to individually volatile assets. However, low-volatility assets do not necessarily have high expected returns. To reach the common 1% monthly target return, the entire RP portfolio must therefore be scaled up substantially. After this rescaling, RP has an annualized volatility of 32.63% and a Sharpe ratio of 0.3679. Thus, although its underlying weighting scheme favors low-volatility assets, it requires substantial overall exposure to achieve the specified return target.

The Equally-Weighted (EW) portfolio ignores differences in both expected returns and risk, initially assigning the same weight to every asset before being rescaled. It achieves the same 12% annualized mean with 27.53% volatility and a 0.4359 Sharpe ratio. Its performance is somewhat better than RP in this sample but it still requires considerably more risk than the MV portfolio to achieve the target return.

The Mean-Variance (MV) portfolio performs substantially better on a risk-adjusted basis. It achieves the same 12% annualized mean excess return with only 8.13% annualized volatility, producing a Sharpe ratio of 1.4758. Unlike EW, which ignores risk differences, or RP, which focuses on individual asset variances, MV uses the assets' expected returns, variances, and covariances. In particular, accounting for covariance allows MV to exploit diversification across assets rather than simply favoring assets that are individually less volatile.

Overall, the results illustrate an important distinction between choosing low-risk assets and constructing a low-risk portfolio. RP favors assets with low individual variance, but that does not guarantee the lowest portfolio variance once the portfolio is scaled to a common expected return. In this sample, MV makes more efficient use of the relationships among the assets and therefore achieves the target return with far less volatility and a substantially higher Sharpe ratio.

***

# 4. EXTRA: Out-of-Sample Performance

### 1. One-step Out-of-Sample (OOS) Performance
Let’s divide the sample to both compute a portfolio and then check its performance out of sample.
* Using only data through the end of `2023`, compute the weights built in Section 3.
* Rescale the weights, (using just the in-sample data,) to set each allocation to have the same mean return of $\mutarg$.
* Using those weights, calculate the portfolio’s Sharpe ratio within that sample.
* Again using those weights, (derived using data through `2023`,) calculate the portfolio’s OOS Sharpe ratio, which is based only on performance in `2024-2025`.

In [112]:
in_sample = excess_returns.loc[:"2023"]
out_sample = excess_returns.loc["2024":]

n = in_sample.shape[1]
ew_weights_is = pd.Series(1 / n, index = in_sample.columns)
ew_weights_is = utils1.scale_weights_to_target(ew_weights_is, in_sample.mean(), TARGET_RETURN)

rp_weights_is = 1 / in_sample.var()
rp_weights_is = utils1.scale_weights_to_target(rp_weights_is, in_sample.mean(), TARGET_RETURN)

mv_weights_is = utils1.tangency_weights(in_sample)
mv_weights_is = utils1.scale_weights_to_target(mv_weights_is, in_sample.mean(), TARGET_RETURN)

ew_returns_is, ew_returns_oos = in_sample @ ew_weights_is, out_sample @ ew_weights_is
rp_returns_is, rp_returns_oos = in_sample @ rp_weights_is, out_sample @ rp_weights_is
mv_returns_is, mv_returns_oos = in_sample @ mv_weights_is, out_sample @ mv_weights_is

ew_mean_is, ew_vol_is = utils1.annualize_stats(ew_returns_is.mean(), ew_returns_is.std())
ew_mean_oos, ew_vol_oos = utils1.annualize_stats(ew_returns_oos.mean(), ew_returns_oos.std())
rp_mean_is, rp_vol_is = utils1.annualize_stats(rp_returns_is.mean(), rp_returns_is.std())
rp_mean_oos, rp_vol_oos = utils1.annualize_stats(rp_returns_oos.mean(), rp_returns_oos.std())
mv_mean_is, mv_vol_is = utils1.annualize_stats(mv_returns_is.mean(), mv_returns_is.std())
mv_mean_oos, mv_vol_oos = utils1.annualize_stats(mv_returns_oos.mean(), mv_returns_oos.std())

oos_comparison = pd.DataFrame({"In-Sample Sharpe": [ew_mean_is / ew_vol_is, rp_mean_is / rp_vol_is, mv_mean_is / mv_vol_is],
                               "Out-of-Sample Sharpe": [ew_mean_oos / ew_vol_oos, rp_mean_oos / rp_vol_oos, mv_mean_oos / mv_vol_oos]}, index = ["EW", "RP", "MV"])

display(oos_comparison)

,In-Sample Sharpe,Out-of-Sample Sharpe
EW,0.3855,0.8525
RP,0.3790,0.2301
MV,1.4551,1.4302


Out-of-sample performance differs across the three allocation methods. 

The MV portfolio is the most stable with its Sharpe ratio decreasing only slightly from 1.4551 in-sample to 1.4302 out-of-sample, suggesting that much of its strong risk-adjusted performance persists in 2024–2025. The EW portfolio actually improves with its Sharpe increasing from 0.3855 to 0.8529, while RP performs substantially worse out-of-sample with its Sharpe falling from 0.3790 to 0.2301. 

Overall, the results show that strong or weak in-sample performance does not necessarily translate directly into future performance.

### 2. Rolling OOS Performance

Iterate the Out-of-Sample performance every year, not just the final year. Namely,
* Start at the end of `2015`, and calculate the weights through that time. Rescale them using the mean returns through that time.
* Apply the weights to the returns in the upcoming year, (`2016`.)
* Step forward a year in time, and recompute.
* Continue until again calculating the weights through `2023` and applying them to the returns in `2024-2025`.

Report the mean, volatility, and Sharpe from this dynamic approach for the following portfolios:
* mean-variance (tangency)
* equally-weighted
* risk-parity
* regularized

In [113]:
ew_oos_returns = []
rp_oos_returns = []
mv_oos_returns = []

for year in range(2015, 2024):
    train = excess_returns.loc[:str(year)] #all information available at the end of this year

    #returns realized after the weights are constructed
    if year < 2023:
        test = excess_returns.loc[str(year + 1)]
    else:
        test = excess_returns.loc["2024":]

    n = train.shape[1]
    ew_weights = pd.Series(1 / n, index = train.columns)
    ew_weights = utils1.scale_weights_to_target(ew_weights, train.mean(), TARGET_RETURN)

    rp_weights = 1 / train.var()
    rp_weights = utils1.scale_weights_to_target(rp_weights, train.mean(), TARGET_RETURN)

    mv_weights = utils1.tangency_weights(train)
    mv_weights = utils1.scale_weights_to_target(mv_weights, train.mean(), TARGET_RETURN)

    ew_oos_returns.append(test @ ew_weights)
    rp_oos_returns.append(test @ rp_weights)
    mv_oos_returns.append(test @ mv_weights)

ew_oos_returns = pd.concat(ew_oos_returns)
rp_oos_returns = pd.concat(rp_oos_returns)
mv_oos_returns = pd.concat(mv_oos_returns)

ew_mean, ew_vol = utils1.annualize_stats(ew_oos_returns.mean(), ew_oos_returns.std())
rp_mean, rp_vol = utils1.annualize_stats(rp_oos_returns.mean(), rp_oos_returns.std())
mv_mean, mv_vol = utils1.annualize_stats(mv_oos_returns.mean(), mv_oos_returns.std())

rolling_oos_stats = pd.DataFrame({"Mean": [ew_mean, rp_mean, mv_mean], 
                                  "Volatility": [ew_vol, rp_vol, mv_vol],
                                  "Sharpe": [ew_mean / ew_vol, rp_mean / rp_vol, mv_mean / mv_vol]}, index = ["EW", "RP", "MV"])
display(rolling_oos_stats)

,Mean,Volatility,Sharpe
EW,0.1878,0.2924,0.6422
RP,0.0823,0.2589,0.3181
MV,0.0786,0.0976,0.8053


In the rolling out-of-sample analysis, EW generates the highest realized mean return at 18.78% but also has the highest volatility at 29.24%, resulting in a Sharpe ratio of 0.6422. RP has lower volatility of 25.89%, but its lower mean return of 8.23% produces the lowest Sharpe ratio of 0.3181. MV achieves the highest risk-adjusted performance, with a 7.86% mean return, 9.76% volatility, and 0.8053 Sharpe ratio. 

Thus, although MV does not generate the highest realized return, it achieves its return with substantially less risk, leading to the strongest Sharpe ratio of the three methods.

***

# 5. EXTRA: Without a Riskless Asset

Re-do Section 2 above, but in the model without a risk-free rate.

That is, build the MV allocation using the two-part formula in the `Mean-Variance` section of the notes.
* This essentially substitutes the risk-free rate with the minimum-variance portfolio.
* Now, the allocation depends nonlinearly on the target mean return, $\mutarg$. (With a risk-free rate, we simply scale the weights up and down to achieve the mean return.)

You will find that, conceptually, the answers are very similar.

***

# 6. EXTRA: Bayesian Allocation

Add the following allocation among the choices in `Section 3`...

#### Regularized (REG)
Much like the Mean-Variance portfolio, set the weights proportional to 

$$\wREG \sim \widehat{\Sigma}^{-1}\mux$$

but this time, use a regularized covariance matrix,

$$\widehat{\Sigma} = \frac{\Sigma + \Sigma_D}{2}$$

where $\Sigma_D$ denotes a *diagonal* matrix of the security variances, with zeros in the off-diagonals.

Thus, $\widehat{\Sigma}$ is obtained from the usual covariance matrix, $\Sigma$, but shrinking all the covariances to half their estimated values.

# 7. EXTRA: Inefficient Tangency

Return to analyzing the excess returns, as in `Section 3`. Include the hedge-fund ETF, QAI.

You might find that the tangency portfolio has a negative mean return. 
- It is on the inefficient portion of the MV frontier.

Calculate the optimal allocation by shorting the tangency portfolio. (That is, multiply the tangency weights by negative one to determine the allocation weights.)

Re-do the analysis of Section 3. Does the addition of `QAI` change much.